# Getting Started with Azure Container Apps Sandboxes

**Sandboxes** are fast, secure, ephemeral compute environments with built-in
suspend/resume — a first-class resource type (`Microsoft.App/SandboxGroups`) in
Azure Container Apps. Unlike *dynamic sessions* (a managed execution experience),
sandboxes give you **direct, programmable control** over isolated compute:
you manage the lifecycle, snapshots, persistent storage, ports, and egress.

This lab walks the full surface end-to-end: create a sandbox group → create a
sandbox from a disk image → exec commands → read/write files → expose a port →
inspect stats → snapshot & restore → build custom disk images → share storage
with volumes → stop/resume → configure auto-suspend/auto-delete lifecycle
policies → lock down egress with allowlists → store and consume secrets → clean up.

## Architecture at a glance

Sandboxes use a **two-plane** architecture, exposed via three Python client classes:

| Client | Plane / Endpoint | Purpose |
|---|---|---|
| `SandboxGroupManagementClient` | ARM control plane (`management.azure.com`) | Create/delete sandbox groups |
| `SandboxGroupClient` | ADC data plane (`management.azuredevcompute.io`) | Create sandboxes, disk images, snapshots, volumes, secrets |
| `SandboxClient` | ADC data plane (per-sandbox) | Exec, files, ports, egress, stop/resume, stats |

## Prerequisites
- An Azure subscription with the **Container Apps SandboxGroup Data Owner** role assigned at the resource group or sandbox group scope.
- Azure CLI [signed in](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively) (`az login`). 
- The `azure-containerapps-sandbox` SDK installed in this notebook's Python environment.

Click **Run All** to execute the full lifecycle, or step through each cell to inspect outputs along the way.


In [ ]:
# Install the SDK and supporting packages (skip if already installed)
%pip install -q --upgrade azure-containerapps-sandbox azure-identity azure-mgmt-resource azure-mgmt-authorization

### 0. Initialize variables and clients

We pull the current Azure subscription from the CLI and build the control-plane clients up front:

- `cli_credential` — an `AzureCliCredential` reused by every other client.
- `resource_mgmt_client` — manages the Azure **resource group** that will host the sandbox group.
- `sandboxgroup_mgmt_client` — ARM control-plane client for creating/deleting **sandbox groups**.

The data-plane `SandboxGroupClient` (used for creating sandboxes, snapshots, volumes, secrets, ...) is built later in step 3 once we know the sandbox group's region.

Adjust `resource_group_name`, `sandbox_group_name`, and `location` if you want to target existing resources or a different region.


In [ ]:
import json, subprocess, sys, time, uuid, urllib.request
from pathlib import Path
from azure.identity import AzureCliCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.containerapps.sandbox import SandboxGroupManagementClient, SandboxGroupClient, endpoint_for_region

# ── Configuration ─────────────────────────────────────────────────────────
lab_name                = Path(globals().get('__vsc_ipynb_file__')).stem
resource_group_name     = 'aca-sandboxes-rg'     # change if using existing Resource Group
sandbox_group_name      = f'lab-{lab_name}'      # change to your desired Sandbox Group name
location                = 'westus3'              # change to your preferred Azure Location
lab_labels              = {'lab': lab_name}      # tag every resource so cleanup can find them

# ── Get current subscription info from Azure CLI (run `az login` first) ───
try:
    proc = subprocess.run('az account show -o json',
        capture_output=True, text=True, check=True, shell=True)
    subscription = json.loads(proc.stdout)
    subscription_id = subscription['id']
except subprocess.CalledProcessError as e:
    sys.exit(f'❌ Azure CLI not installed or not signed in. Run `az login` first.\n{e.stderr}')

cli_credential = AzureCliCredential() # reused by every Azure client below
resource_mgmt_client = ResourceManagementClient(cli_credential, subscription_id) # manage resource groups
sandboxgroup_mgmt_client = SandboxGroupManagementClient(cli_credential, subscription_id=subscription_id, resource_group=resource_group_name) # manage sandbox groups

print(f'🧪 Lab:                   {lab_name}')
print(f'👤 User:                  {subscription["user"]["name"]}')
print(f'🏛️ Tenant Id:             {subscription["tenantId"]}')
print(f'🔑 Subscription:          {subscription["name"]} ({subscription_id})')
print(f'🌍 Target Azure Region:   {location}')
print(f'📁 Target Resource Group: {resource_group_name}')
print(f'📦 Target Sandbox Group:  {sandbox_group_name}')


### 1. Create the resource group and sandbox group

A **sandbox group** is the top-level management boundary — all sandboxes, disk
images, snapshots, volumes, and secrets are scoped to it. Use sandbox groups to
organize compute by application, team, or environment.

This cell creates (or reuses) both the Azure resource group and the sandbox
group inside it. The sandbox group is an ARM resource, so it goes through the
control plane (`management.azure.com`).


In [ ]:
if resource_mgmt_client.resource_groups.check_existence(resource_group_name):
    resource_group = resource_mgmt_client.resource_groups.get(resource_group_name)
    print(f'♻️ Using existing resource group: {resource_group.name} ({resource_group.location})')
else:
    resource_group = resource_mgmt_client.resource_groups.create_or_update(
        resource_group_name, {'location': location, 'tags': lab_labels})
    print(f'✅ Created resource group: {resource_group.name} ({resource_group.location})')

# Reuse the sandbox group if it exists; otherwise create it tagged with the lab name.
existing = next((g for g in sandboxgroup_mgmt_client.list_groups() if g.name == sandbox_group_name), None)
if existing:
    action, icon = 'Using existing', '♻️'
else:
    sandboxgroup_mgmt_client.create_group(
        sandbox_group_name, location=resource_group.location, tags=lab_labels)
    action, icon = 'Created', '✅'

# Refetch to get a fully-hydrated object regardless of which branch ran above.
sandboxgroup = sandboxgroup_mgmt_client.get_group(sandbox_group_name)

sandbox_group_url = f'https://sandboxes.azure.com/sandbox-groups/{subscription_id}/{resource_group_name}/{sandbox_group_name}/sandboxes'
print(f'{icon} {action} sandbox group: {sandboxgroup.name}')
print(f'  🆔 Id:                 {sandboxgroup.id}')
print(f'  🌍 Location:           {sandboxgroup.location}')
print(f'  ⚙️ Provisioning state: {sandboxgroup.properties["provisioningState"]}')
print(f'  🏷️ Tags:               {sandboxgroup.tags or "{}"}')
print(f'  🔗 Portal:             {sandbox_group_url}')


### 2. Grant yourself data-plane access

Creating the sandbox group via ARM (the control plane) doesn't automatically let you manage sandboxes inside it — that happens on the **data plane**, which is guarded by a separate role. We assign **Container Apps SandboxGroup Data Owner** to your signed-in user at the resource group scope so the next steps can create and manage sandboxes.

> Role assignments can take 30–60 seconds to propagate. The SDK's HTTP pipeline retries `403`s during this window.


In [ ]:
from azure.mgmt.authorization import AuthorizationManagementClient

# Grant YOURSELF (the signed-in user) the data-plane role so this notebook can create sandboxes.
# Without this, the data-plane SandboxGroupClient calls in the next steps would return 403.
auth_client = AuthorizationManagementClient(cli_credential, subscription_id)
role_name = 'Container Apps SandboxGroup Data Owner'
scope = f'/subscriptions/{subscription_id}/resourceGroups/{resource_group_name}' # can be as narrow as a single sandbox group
role_def = next(auth_client.role_definitions.list(scope, filter=f"roleName eq '{role_name}'"))

# Resolve the signed-in user's object id from the Azure CLI.
proc = subprocess.run('az ad signed-in-user show --query id -o tsv',
    capture_output=True, text=True, check=True, shell=True)
user_principal_id = proc.stdout.strip()

try:
    auth_client.role_assignments.create(scope, uuid.uuid4(), {
        'role_definition_id': role_def.id,
        'principal_id': user_principal_id,
        'principal_type': 'User',
    })
    print(f"✅ Assigned '{role_name}' to you ({subscription['user']['name']})")
    print('  ⏳ May take 30–60s to propagate before you can create sandboxes.')
except Exception as ex:
    if 'RoleAssignmentExists' in str(ex) or 'Conflict' in str(ex):
        print(f"♻️ '{role_name}' already assigned to you ({subscription['user']['name']})")
    else:
        raise

print(f'  🆔 Role definition:  {role_def.id}')
print(f'  👤 User principalId: {user_principal_id}')
print(f'  🎯 Scope:            {scope}')


### 3. Create a sandbox from a public disk image

Now we switch to the **data plane** by building a `SandboxGroupClient` pointed at the regional ADC endpoint. From here we can list available **public disk images** (OCI container images preconverted for use as sandbox root filesystems) and launch a new sandbox.

`begin_create_sandbox` is a long-running operation: calling `.result()` blocks until the sandbox reaches the **Running** state, then returns a `SandboxClient` bound to it. Optional kwargs include `cpu`, `memory`, `labels`, `environment`, and `auto_suspend_seconds`.


In [ ]:
client = SandboxGroupClient(endpoint_for_region(sandboxgroup.location), cli_credential, subscription_id=subscription_id, resource_group=resource_group_name, sandbox_group=sandbox_group_name) # manage sandboxes

public_disk_images = list(client.list_public_disk_images())
print(f'📚 Available public disk images: {", ".join(image.name for image in public_disk_images)}')

# Tag the sandbox with the lab label so the cleanup cell at the bottom can find it.
sandbox = client.begin_create_sandbox(disk='ubuntu', labels={**lab_labels, 'env': 'development'}).result()

# Print key sandbox details.
sandbox_details = sandbox.get()
sandbox_url = f'https://sandboxes.azure.com/sandbox-groups/{subscription_id}/{resource_group_name}/{sandbox_group_name}/sandboxes/{sandbox_details.id}'
print('\n📦 Details for the sandbox created:')
print(f'  🆔 ID             : {sandbox_details.id}')
print(f'  🟢 State          : {sandbox_details.state}')
print(f'  🌍 Region         : {sandbox_details.region}')
print(f'  🖥️ VMM type       : {sandbox_details.vmm_type}')
print(f'  💿 Disk image     : {sandbox_details.sources_ref.disk_image if sandbox_details.sources_ref else None}')
print(f'  🕒 Created at     : {sandbox_details.created_at}')
print(f'  ⚙️ CPU            : {sandbox_details.resources.cpu if sandbox_details.resources else None}')
print(f'  🧠 Memory         : {sandbox_details.resources.memory if sandbox_details.resources else None}')
print(f"  🏷️ Labels         : {sandbox_details.labels or '{}'}")
print(f'  🔗 Portal         : {sandbox_url}')


### 4. Execute commands

`sandbox.exec(...)` runs a shell command inside the sandbox and returns an `ExecResult` with `exit_code`, `stdout`, and `stderr`. Commands can be chained with `&&`/`||` and exit codes propagate back — useful for scripting agent workflows or CI steps that need to react to failures.


In [ ]:
result = sandbox.exec('whoami && uname -a')
print(f'🔢 exit code: {result.exit_code}')
if result.stdout:
    print(f'📤 stdout: {result.stdout}')
if result.stderr:
    print(f'⚠️ stderr: {result.stderr}')


### 5. Write and read files

File operations work directly against the sandbox's filesystem — no SSH or volume mount required. Beyond `write_file`/`read_file`, the SDK also exposes `list_files`, `stat_file`, `mkdir`, and `delete_file`.

Here we drop an `index.html` that the next step will serve over HTTP.


In [ ]:
file_path = 'index.html'
file_content = '<html><body><h1>Hello from Azure Container Apps sandboxes!</h1></body></html>'

# Write
sandbox.write_file(file_path, file_content)
print(f'📝 Wrote {len(file_content)} bytes to {file_path}')

# Stat — verify size and that it's a file, not a directory
stat = sandbox.stat_file(file_path)
print(f'📊 stat: size={stat.size} bytes, is_directory={stat.is_directory}')

# Read back
content = sandbox.read_file(file_path)
print(f'📖 Read from {file_path}: {content.decode()}')
assert file_content in content.decode()

### 6. Expose a port with a public URL

`sandbox.add_port(port, anonymous=True)` publishes an ingress endpoint to a process running inside the sandbox and returns a public HTTPS URL. We first start Python's built-in HTTP server in the background (serving the `index.html` we just wrote), then expose port 8080. Open the printed URL in a browser to see the page served straight from the sandbox.

Use `sandbox.remove_port(port)` or `sandbox.update_ports([...])` to change exposure later. Combine with **egress policies** (`set_egress_default`, `add_egress_host_rule`) to control outbound traffic.


In [ ]:
# Serve index.html on the specified port
port_number = 8080
result = sandbox.exec(f'nohup python3 -m http.server {port_number} >/tmp/http.log 2>&1 &')
print(f'🔢 exit code: {result.exit_code}')

port = sandbox.add_port(port_number, anonymous=True)
print(f'🌐 Public URL: {port.url}')

# Poll the public URL — proxy + in-sandbox server can take a few seconds to be reachable.
last_err = None
for attempt in range(10):
    try:
        with urllib.request.urlopen(port.url, timeout=5) as resp:
            body = resp.read().decode()
        print(f'✅ HTTP {resp.status} (after {attempt + 1} attempt[s]) — content from the web: {body}')
        assert file_content in body
        break
    except Exception as ex:
        last_err = ex
        time.sleep(1)
else:
    print(f'⚠️ Could not reach {port.url}: {last_err}')
    print('   Check the in-sandbox server log: sandbox.exec("cat /tmp/http.log")')


### 7. Inspect runtime stats

`sandbox.get_stats()` returns live resource usage for the sandbox: CPU (in nano-cores), memory, storage, and network rx/tx bytes. Use this for dashboards, autoscaling decisions, or to verify that a workload is consuming the resources you expect from its tier (XS/S/M/L).


In [ ]:
# Sandbox resource usage
stats = sandbox.get_stats()

# CPU — reported in nano-cores (1 core = 1_000_000_000 nano-cores)
cpu = stats.cpu
cores = (cpu.usage_nano_cores or 0) / 1_000_000_000 if cpu else 0
print(f'⚙️ CPU:     {cores:.3f} cores ({cores * 1000:.0f} mCPU)')

# Memory
mem = stats.memory
used_mb  = (mem.used_bytes  or 0) // 1024 // 1024 if mem else 0
total_mb = (mem.total_bytes or 0) // 1024 // 1024 if mem else 0
mem_pct = (used_mb / total_mb * 100) if total_mb else 0
print(f'🧠 Memory:  {used_mb}MB / {total_mb}MB ({mem_pct:.1f}%)')

# Storage
storage = stats.storage
if storage:
    used_gb  = (storage.used_bytes  or 0) / 1024 / 1024 / 1024
    total_gb = (storage.total_bytes or 0) / 1024 / 1024 / 1024
    storage_pct = (used_gb / total_gb * 100) if total_gb else 0
    print(f'💿 Storage: {used_gb:.2f}GB / {total_gb:.2f}GB ({storage_pct:.1f}%)')
else:
    print('💿 Storage: (no data)')

# Network
net = stats.network
rx_mb = (net.rx_bytes or 0) / 1024 / 1024 if net else 0
tx_mb = (net.tx_bytes or 0) / 1024 / 1024 if net else 0
print(f'🌐 Network: rx {rx_mb:.2f}MB / tx {tx_mb:.2f}MB')


### 8. Snapshots — capture and restore full state

**Snapshots** capture the full state of a running sandbox, including memory and disk. They're the foundation for:

- **Suspend & resume** — pause and restore with all processes/data intact.
- **Clone environments** — spin up new sandboxes from a known-good baseline.
- **Share baselines** — distribute preconfigured environments across a team.

This cell creates a snapshot, lists all snapshots in the group, then uses `begin_create_sandbox(snapshot_id=...)` to launch a **brand new sandbox** restored from that snapshot.


In [ ]:
# Create a snapshot of the current sandbox state
snapshot = sandbox.create_snapshot(name='getting-started', labels=lab_labels)
print(f'📸 Snapshot created: {snapshot.id}')
print(f'  🕒 Created at: {snapshot.created_at_utc}')

# List all snapshots in the sandbox group
snapshots = list(client.list_snapshots())
print(f'\n📚 Total snapshots in group: {len(snapshots)}')
for s in snapshots:
    print(f'  - {s.id} (sandbox={s.sandbox_id})')

# Restore — create a new sandbox from the snapshot
restored = client.begin_create_sandbox(snapshot_id=snapshot.id).result()
print(f'\n♻️ Restored sandbox from snapshot: {restored.sandbox_id}')

# Snapshot-restored sandboxes need a few extra seconds to warm up before exec/file ops.
restored.wait_for_running(timeout=120)
verify = restored.exec('echo restored')
print(f'✅ Verified restored sandbox: {(verify.stdout or "").strip()}')

# Clean up: delete the restored sandbox and the snapshot
client.delete_sandbox(restored.sandbox_id)
client.delete_snapshot(snapshot.id)
print(f'🗑️ Deleted restored sandbox and snapshot {snapshot.id}')


### 9. Disk images — bring your own root filesystem

Disk images are OCI container images converted for use as sandbox root filesystems. You can build private disk images from:

- **A running sandbox** — `sandbox.commit(name=...)` packages the current state.
- **A container registry** — `client.create_disk_image("docker.io/...", name=...)` pulls and converts any public or private image.
- **A Dockerfile** — build custom images for repeatable environments.

Once created, launch sandboxes from them via `begin_create_sandbox(disk_id=...)`. Use `list_disk_images` / `get_disk_image` / `delete_disk_image` to manage them.


In [ ]:
# Commit current sandbox state as a private disk image
disk_image_from_snapshot = sandbox.commit(name='my-custom-image')
print(f'💾 Committed sandbox to disk image: {disk_image_from_snapshot.id}')

# Create a new sandbox from the committed disk image
sandbox_from_committed_disk_image = client.begin_create_sandbox(disk_id=disk_image_from_snapshot.id, labels=lab_labels).result()
print(f'🚀 Created sandbox from committed disk: {sandbox_from_committed_disk_image.sandbox_id}')

# Create a disk image directly from a container image
disk_image_from_container = client.create_disk_image('docker.io/library/ubuntu:22.04', name='ubuntu-22')
print(f'📦 Created disk image from container: {disk_image_from_container.id}')

# Create a new sandbox from the container image
sandbox_from_container_image = client.begin_create_sandbox(disk_id=disk_image_from_container.id, labels=lab_labels).result()
print(f'🚀 Created sandbox from container image: {sandbox_from_container_image.sandbox_id}')

# List private disk images
private_disk_images = list(client.list_disk_images())
print(f'\n📚 Available private disk images: {", ".join(image.name or image.id for image in private_disk_images)}')

# Delete the two transient sandboxes we just spun up
client.delete_sandbox(sandbox_from_committed_disk_image.sandbox_id)
client.delete_sandbox(sandbox_from_container_image.sandbox_id)
print('🗑️ Deleted transient disk-image sandboxes')

# Delete disk images
client.delete_disk_image(disk_image_from_snapshot.id)
client.delete_disk_image(disk_image_from_container.id)
print('🗑️ Deleted disk images')


### 10. Volumes

**Volumes** are persistent, group-scoped storage that can be mounted into one or more sandboxes at the same time. Unlike a sandbox's root disk (which lives and dies with the sandbox) or a snapshot (which is a point-in-time copy), a volume:

- **Persists** beyond any individual sandbox — it's owned by the sandbox group.
- **Is shareable** — multiple sandboxes can mount the same volume concurrently to exchange files or share state.
- **Is backed by Azure Storage** (e.g. `AzureBlob`), so data is durable and visible outside the sandbox plane.

Typical uses: passing artifacts between a producer and a consumer sandbox, hosting a shared dataset across a fleet of worker sandboxes, or persisting agent state across runs.

Volumes are created on the `SandboxGroupClient` (`create_volume` / `list_volumes` / `get_volume` / `delete_volume`) and attached to a sandbox via `sandbox.add_volume_mount(AddVolumeMountRequest(volume_name=..., mountpoint=...))`.

The cell below creates a blob-backed volume, spins up a **producer** sandbox that writes a JSON file to `/mnt/shared`, then spins up a separate **consumer** sandbox that reads the same file — proving cross-sandbox shared state — and cleans everything up.


In [ ]:
from azure.containerapps.sandbox import AddVolumeMountRequest

# Create a shared Azure Blob volume and mount it in two sandboxes.
volume_name = f'vol-{uuid.uuid4().hex[:8]}'
client.create_volume(volume_name, type='AzureBlob')
mount = AddVolumeMountRequest(volume_name=volume_name, mountpoint='/mnt/shared')
print(f'📁 Created volume: {volume_name}')

try:
    producer = client.begin_create_sandbox(labels={**lab_labels, 'role': 'producer'}).result()
    producer.add_volume_mount(mount)
    producer.exec('echo \'{"answer":42}\' > /mnt/shared/output.json')

    consumer = client.begin_create_sandbox(labels={**lab_labels, 'role': 'consumer'}).result()
    consumer.add_volume_mount(mount)
    read = consumer.exec('cat /mnt/shared/output.json').stdout.strip()
    print(f'✍️  Producer {producer.sandbox_id} wrote -> 📖 Consumer {consumer.sandbox_id} read: {read}')
    assert '42' in read
finally:
    for sbx in (producer, consumer):
        client.delete_sandbox(sbx.sandbox_id)
    client.delete_volume(volume_name)
    print(f'🗑️ Cleaned up sandboxes and volume {volume_name}')


### 11. Stop and resume

Sandboxes transition through lifecycle states: **Running**, **Suspended**, **Idle**, **Stopped**, plus transient states (`Resuming`, `Creating`, `Stopping`, `Deleting`). You can drive transitions explicitly with `sandbox.stop()` / `sandbox.resume()`.

Stopped sandboxes preserve their disk state — resuming brings the workload back online without recreating anything.


In [ ]:
# Stop the sandbox (pauses execution, preserves state)
sandbox.stop()
print(f'⏸️ Stopped sandbox: {sandbox.sandbox_id}  new state={client.get_sandbox(sandbox.sandbox_id).state}')

# Resume and wait deterministically for Running (no time.sleep race).
sandbox.resume()
sandbox.wait_for_running(timeout=120)
print(f'▶️ Resumed sandbox: {sandbox.sandbox_id}  new state={client.get_sandbox(sandbox.sandbox_id).state}')

# ensure_running() is a no-op when already Running — safe to call before exec/file ops.
sandbox.ensure_running()
assert 'Running' in client.get_sandbox(sandbox.sandbox_id).state
result = sandbox.exec('echo Back from suspend!')
print(f'📤 stdout: {result.stdout.strip()}')


### 12. Configure lifecycle policies

Manually calling `stop()` and `resume()` works, but in production you usually want the platform to manage idle compute for you. A **lifecycle policy** attaches automatic rules to a sandbox:

- **`AutoSuspendPolicy`** — suspend the sandbox after `interval` seconds of inactivity. `mode='Memory'` keeps RAM checkpointed so resume is fast; `mode='Disk'` is cheaper but slower to wake.
- **`AutoDeletePolicy`** — permanently delete the sandbox after `delete_interval_seconds` of being suspended/idle. Use this as a hard backstop against runaway cost.

Policies are applied per-sandbox with `sandbox.set_lifecycle_policy(...)`. The cell below sets an aggressive 10-second auto-suspend so you can watch the state transition from **Running** to **Idle** without waiting around — production values would typically be minutes or hours.


In [ ]:
from azure.containerapps.sandbox import LifecyclePolicy, AutoDeletePolicy, AutoSuspendPolicy

# Apply: suspend after 10s idle, delete 10 min after that.
sandbox.set_lifecycle_policy(LifecyclePolicy(
    auto_suspend=AutoSuspendPolicy(enabled=True, interval=10, mode='Memory'),
    auto_delete=AutoDeletePolicy(enabled=True, delete_interval_seconds=600),
))
print(f'⚙️ Policy applied — state now: {client.get_sandbox(sandbox.sandbox_id).state}')

# Poll until the platform suspends the idle sandbox (or we give up after ~60s).
for _ in range(60):
    state = client.get_sandbox(sandbox.sandbox_id).state
    if state != 'Running':
        break
    time.sleep(1)
print(f'💤 Final state after suspend: {state}')
assert 'Idle' in state

# Resume
sandbox.resume()
sandbox.wait_for_running(timeout=120)
print(f'▶️ Resumed - state now: {client.get_sandbox(sandbox.sandbox_id).state}')


### 13. Egress policies

By default, sandboxes can reach any public host. For untrusted workloads (agents running model-generated code, third-party plugins, build steps) you'll usually want to **lock that down to an allowlist**.

The egress API is rule-based:

- **`sandbox.set_egress_default('Deny' | 'Allow')`** — the fallback action when no host rule matches.
- **`sandbox.add_egress_host_rule(pattern, action=...)`** — pin specific hosts (supports wildcards like `*.github.com`) to `Allow` or `Deny`. You can also pass a full `EgressPolicy(...)` via `set_egress_policy(...)`.
- **`sandbox.get_egress_policy()`** — read the current policy.
- **`sandbox.get_egress_decisions()`** — audit log of recent allow/deny decisions for forensic review.

Policy changes propagate to the data-plane proxy asynchronously (a few seconds), so the cell below polls for enforcement instead of trusting a single check. The pattern: switch default to `Deny`, allowlist `*.github.com`, then prove that an arbitrary host is blocked while github is still reachable.


In [ ]:
def http_status(host):
    r = sandbox.exec(f"curl -sS -o /dev/null -w '%{{http_code}}' --max-time 8 https://{host} || echo FAIL")
    return r.stdout.strip() or 'FAIL'

sandbox.ensure_running()
print(f'🌐 Baseline example.com -> {http_status("example.com")}')

sandbox.set_egress_default('Deny')
policy = sandbox.add_egress_host_rule('*.github.com', action='Allow')
print(f'🔒 Policy: default={policy.default_action}, rules={[(r.pattern, r.action) for r in policy.host_rules]}')

# Poll until policy propagates to the data-plane proxy.
for attempt in range(15):
    if http_status('example.com') != '200': break
    time.sleep(1)
print(f'⏱️ Enforcement active after ~{attempt + 1}s')

example_http_status = http_status('example.com')
assert '403' in example_http_status
print(f'🚫 example.com (denied)    -> {example_http_status}')

github_http_status = http_status('api.github.com')
assert '200' in github_http_status
print(f'✅ api.github.com (allowed) -> {github_http_status}')

# Poll the audit log — proxy logs propagate asynchronously too.
for attempt in range(20):
    net = sandbox.get_egress_decisions().network_egress
    if net.allowed or net.denied: break
    time.sleep(1)

print(f'\n📜 Audit log (after ~{attempt + 1}s): {len(net.allowed)} allowed, {len(net.denied)} denied')
for tag, entries in (('✅', net.allowed), ('🚫', net.denied)):
    for e in entries:
        print(f'  {tag} {e.timestamp}  {e.method} {e.scheme}://{e.host}{e.path}')


### 14. Secrets

**Secrets** are sandbox-group-scoped key/value bundles, durably stored by the
platform and injected into sandboxes as environment variables — no need to bake
credentials into disk images or pass them through `exec` arguments where they'd
land in shell history.

Each secret has:

- **`id`** — a stable name unique within the sandbox group.
- **`values`** — a dict of key/value pairs (e.g. `{'API_KEY': '...', 'MODEL': '...'}`).

The API is intentionally small:

| Method | Purpose |
|---|---|
| `client.upsert_secret(id, values)` | Create or replace a secret (idempotent). |
| `client.list_secrets()` | List all secret ids in the group. |
| `client.list_secret_keys(id)` | List a secret's keys without revealing values. |
| `client.peek_secret(id)` | Return the actual values — handle with care. |
| `client.delete_secret(id)` | Remove the secret from the group. |

To consume a secret inside a sandbox, reference it at creation time via the
`environment` / `secret_refs` kwargs on `begin_create_sandbox` — the platform
mounts the resolved values into the process environment.

The cell below walks the full CRUD loop: create → list → introspect keys →
peek (masked) → update → clean up.



In [ ]:
secret_id = f'demo-{uuid.uuid4().hex[:8]}'

# Create the secret (group-scoped: any sandbox in the group can read it).
client.upsert_secret(secret_id, {'API_KEY': 'sk-test-123', 'MODEL': 'gpt-4'})
print(f'🔐 Upserted secret: {secret_id}')

# List — confirm the new secret is visible in the group.
names = [s.id for s in client.list_secrets()]
print(f'📚 {len(names)} secret(s) in group; demo present = {secret_id in names}')

# Keys are returned in cleartext; values require peek_secret.
print(f'🗝️ Keys: {client.list_secret_keys(secret_id)}')

# Peek — returns values; mask before printing so we don't leak them into the notebook.
peek = client.peek_secret(secret_id)
for k, v in (peek.values or {}).items():
    print(f'  {k} = {(v[:3] + "***") if v else ""}')

# Update — upsert_secret is idempotent; same id replaces the value.
client.upsert_secret(secret_id, {'API_KEY': 'sk-updated-456', 'MODEL': 'gpt-4o'})
peek = client.peek_secret(secret_id)
print(f'♻️ API_KEY now starts with {peek.values["API_KEY"][:6]}...')

# Clean up so re-runs of the lab stay tidy.
client.delete_secret(secret_id)
print(f'🗑️ Deleted secret: {secret_id}')


### 15. Clean up

We use the **lab label** (`{'lab': '01-getting-started'}`) we attached to every sandbox throughout this notebook to find and delete them in one pass — that way leftover sandboxes from a previous run (or a crashed kernel) get cleaned up too, not just the `sandbox` variable in scope.

Then we delete the sandbox group, which cascades and removes any remaining sandboxes, disk images, snapshots, volumes, and secrets scoped to it.

The resource group delete is left commented out so you don't accidentally remove a shared RG — uncomment it if you created the RG just for this lab.



In [ ]:
# Find every sandbox we tagged with this lab's label — survives kernel restarts.
lab_sandboxes = list(client.list_sandboxes(labels=lab_labels))
print(f'🔎 Found {len(lab_sandboxes)} sandbox(es) tagged {lab_labels}')
for s in lab_sandboxes:
    client.delete_sandbox(s.id)
    print(f'  🗑️ Deleted sandbox: {s.id}')

# Delete the sandbox group — removes any remaining sandboxes, disk images, snapshots, volumes, and secrets scoped to it.
sandboxgroup_mgmt_client.delete_group(sandbox_group_name)
print(f'🗑️ Deleted sandbox group: {sandbox_group_name}')

# Optionally delete the resource group (fire-and-forget — runs in the background).
# resource_mgmt_client.resource_groups.begin_delete(resource_group_name)
# print(f'🗑️ Deleting resource group (async): {resource_group_name}')



## Troubleshooting

Common issues and how to resolve them:

- **`403 Forbidden` / `AuthorizationFailed`** — your principal is missing the **Container Apps SandboxGroup Data Owner** role at the resource group or sandbox group scope. Assign it via the Azure portal or:
  ```bash
  az role assignment create --assignee "<your-upn-or-objectid>" \
    --role "Container Apps SandboxGroup Data Owner" \
    --scope "/subscriptions/<sub-id>/resourceGroups/<rg-name>"
  ```
  Role assignments take 30–60 seconds to propagate; the SDK auto-retries 403s during that window.

- **`Azure CLI not installed or signed in`** — run `az login` (Entra ID account only — personal Microsoft accounts aren't supported) and re-run the first cell.

- **`SubscriptionNotRegistered` / `Microsoft.App not registered`** — register the provider once per subscription:
  ```bash
  az provider register --namespace Microsoft.App
  ```

- **Region not available** — sandboxes are in early access in a limited set of regions. If `endpoint_for_region(location)` raises or `create_group` fails, try `eastus2`, `westus3`, or another supported region.

- **Sandbox stuck in `Creating` / LRO timeout** — long-running operations expose `.result(timeout=...)`. Retry, or call `client.list_sandboxes()` to inspect state. Delete stuck sandboxes via `client.delete_sandbox(sandbox_id)`.

- **`add_port` URL returns 502 / connection refused** — make sure the in-sandbox process is actually listening on the port *before* `add_port` is called, and that it binds to `0.0.0.0` (not `127.0.0.1`). Check the server's log: `sandbox.exec('cat /tmp/http.log')`.

- **Snapshot / commit fails with "sandbox not running"** — `create_snapshot` and `commit` require the sandbox in **Running** state. Call `sandbox.resume()` first if it was stopped or suspended.

- **Quota exceeded when creating multiple sandboxes** — each sandbox group has per-region capacity limits during early access. Delete unused sandboxes or request additional quota via your Microsoft contact.

- **Need more diagnostics** — enable HTTP logging to see request/response details:
  ```python
  import logging
  logging.basicConfig(level=logging.DEBUG)
  logging.getLogger('azure').setLevel(logging.DEBUG)
  ```


## Next steps

This lab covered the core lifecycle. The full sample set in
[`python/samples/00-get-started`](../samples/00-get-started/) goes deeper into each area:

- **[00-sandbox-groups](../samples/00-get-started/00-sandbox-groups/)** — creating and managing sandbox groups via ARM.
- **[01-sandboxes](../samples/00-get-started/01-sandboxes/)** — sandbox creation, listing, and inspection.
- **[02-snapshots](../samples/00-get-started/02-snapshots/)** — capturing, listing, and restoring snapshots.
- **[03-disks](../samples/00-get-started/03-disks/)** — building and managing private disk images.
- **[04-volumes](../samples/00-get-started/04-volumes/)** — persistent storage shared across sandboxes.
- **[05-lifecycle](../samples/00-get-started/05-lifecycle/)** — auto-suspend / auto-delete policies and deterministic state waits.
- **[06-ports](../samples/00-get-started/06-ports/)** — exposing in-sandbox services via public URLs.
- **[07-files](../samples/00-get-started/07-files/)** — read/write/stat/list/mkdir/delete against the sandbox filesystem.
- **[08-egress](../samples/00-get-started/08-egress/)** — controlling outbound network traffic from sandboxes.
- **[09-secrets](../samples/00-get-started/09-secrets/)** — sandbox-scoped secrets.
- **[10-identity](../samples/00-get-started/10-identity/)** — managed identity for sandboxes.
- **[11-labels](../samples/00-get-started/11-labels/)** — using labels to organize and filter resources.
- **[12-interactive-shell](../samples/00-get-started/12-interactive-shell/)** — attaching an interactive shell to a sandbox.
- **[01-webapps/simple-anonymous](../samples/01-webapps/simple-anonymous/)** — a runnable end-to-end web app sample.
